In [ ]:
# -*- coding: utf-8 -*-
"""
Algoritmo de Treinamento Stable Diffusion (LoRA) - VERSÃO HUGGING FACE OFFICIAL
Autor: Gemini
Para: Cairo
Data: 2025-06-17 (Atualizado com Limpeza de Disco)

DESCRIÇÃO:
Esta versão inclui uma rotina de saneamento de disco para remover resíduos
de tentativas falhas anteriores (Kohya, modelos duplicados) antes de iniciar.
Utiliza a biblioteca oficial 'Diffusers' da Hugging Face.

INSTRUÇÕES:
1. Faça upload do 'imagens.zip'.
2. Defina o 'trigger_word'.
3. Execute. (O script limpará o disco automaticamente antes de começar).
"""

import os
import subprocess
import shutil
import zipfile

# --- 1. CONFIGURAÇÃO (Engenharia do Processo) ---

# Palavra-gatilho (Como você chamará o modelo no prompt: "photo of cairo person")
trigger_word = "lol"

# Arquivo de entrada
input_zip_file = "imagens.zip"

# Diretórios
root_dir = "/content"
output_dir = os.path.join(root_dir, "output_lora")
train_data_dir = os.path.join(root_dir, "instance_data")
repo_dir = os.path.join(root_dir, "diffusers")

# Modelo Base (SD 1.5 continua sendo o melhor para LoRA rápido)
model_name = "runwayml/stable-diffusion-v1-5"

# --- 2. FUNÇÕES DE INFRAESTRUTURA ---

def run_command(command, message):
    print(f"⚙️ [INICIANDO]: {message}...")
    try:
        subprocess.run(command, shell=True, check=True)
        print(f"✅ [SUCESSO]: {message}\n")
    except subprocess.CalledProcessError as e:
        print(f"❌ [ERRO CRÍTICO]: {message} falhou. Erro: {e}")
        raise

def deep_cleanup():
    """
    Remove arquivos de tentativas anteriores para liberar espaço em disco.
    """
    print("--- 0. LIMPEZA DE DISCO (DEEP CLEAN) ---")

    # Lista de pastas criadas por tentativas anteriores (Kohya e outros scripts)
    trash_folders = [
        "/content/kohya-ss",          # Script anterior
        "/content/pretrained_model",  # Modelo baixado via aria2
        "/content/train_data",        # Dados antigos
        "/content/sample_data",       # Dados inúteis do Colab
        "/content/temp_extract"       # Lixo temporário
    ]

    freed_space = False
    for folder in trash_folders:
        if os.path.exists(folder):
            print(f"🗑️ Removendo lixo encontrado: {folder}")
            shutil.rmtree(folder)
            freed_space = True

    # Limpa cache do PIP
    run_command("pip cache purge", "Limpando Cache do PIP")

    if freed_space:
        print("✅ Espaço em disco recuperado.")
    else:
        print("✅ Disco já estava limpo.")

def check_gpu():
    print("--- 1. DIAGNÓSTICO DE HARDWARE ---")
    try:
        subprocess.run("nvidia-smi", shell=True)
    except:
        print("⚠️ GPU não detectada. O treino será impossível.")

def setup_environment():
    print("--- 2. INSTALANDO BIBLIOTECAS (MODO COMPATIBILIDADE) ---")

    if not os.path.exists(repo_dir):
        run_command(f"git clone https://github.com/huggingface/diffusers {repo_dir}", "Clonando Diffusers")

    dependencies = (
        "pip install -U -q "
        "accelerate "
        "transformers "
        "ftfy "
        "bitsandbytes "
        "peft "
        "torch "
        "torchvision "
        "--break-system-packages"
    )
    run_command(dependencies, "Instalando Dependências Core")
    run_command(f"pip install -e {repo_dir} --break-system-packages", "Instalando Diffusers (Source)")

def prepare_dataset():
    print("--- 3. PREPARANDO IMAGENS ---")

    if os.path.exists(train_data_dir):
        shutil.rmtree(train_data_dir)
    os.makedirs(train_data_dir, exist_ok=True)

    zip_path = os.path.join(root_dir, input_zip_file)
    if not os.path.exists(zip_path):
        print(f"🛑 Arquivo {input_zip_file} não encontrado.")
        return False

    print(f"📂 Extraindo {input_zip_file}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        temp_dir = os.path.join(root_dir, "temp_extract_new")
        zip_ref.extractall(temp_dir)

        count = 0
        allowed = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

        for root, dirs, files in os.walk(temp_dir):
            for file in files:
                if os.path.splitext(file)[1].lower() in allowed:
                    src = os.path.join(root, file)
                    dst = os.path.join(train_data_dir, f"img_{count}.jpg")
                    shutil.move(src, dst)
                    count += 1

        shutil.rmtree(temp_dir)
        print(f"✅ {count} imagens prontas em {train_data_dir}")
        return True

def train():
    print("--- 4. TREINANDO (Hugging Face Dreambooth LoRA) ---")

    # Tenta localizar o script no repo clonado
    script_paths = [
        os.path.join(repo_dir, "examples", "dreambooth", "train_dreambooth_lora.py"),
        os.path.join(repo_dir, "examples", "research_projects", "dreambooth_lora", "train_dreambooth_lora.py")
    ]

    script_path = "train_dreambooth_lora.py" # Default fallback
    found = False
    for path in script_paths:
        if os.path.exists(path):
            script_path = path
            found = True
            break

    if not found:
        print("⚠️ Script local não encontrado, baixando versão raw...")
        run_command("wget https://raw.githubusercontent.com/huggingface/diffusers/main/examples/dreambooth/train_dreambooth_lora.py", "Baixando Script")

    args = (
        f"accelerate launch {script_path} "
        f"--pretrained_model_name_or_path='{model_name}' "
        f"--instance_data_dir='{train_data_dir}' "
        f"--output_dir='{output_dir}' "
        f"--instance_prompt='photo of {trigger_word}' "
        f"--resolution=512 "
        f"--train_batch_size=1 "
        f"--gradient_accumulation_steps=1 "
        f"--checkpointing_steps=500 "
        f"--learning_rate=1e-4 "
        f"--lr_scheduler='constant' "
        f"--lr_warmup_steps=0 "
        f"--max_train_steps=1000 "
        f"--use_8bit_adam "
        f"--mixed_precision='fp16' "
    )

    print("🚀 Iniciando Neural Network...")
    run_command(args, "Treinamento")

def main():
    deep_cleanup() # Limpeza antes de tudo
    check_gpu()
    setup_environment()
    if prepare_dataset():
        train()
        print("\n=======================================================")
        print(f"🎉 SUCESSO! Modelo salvo em: {output_dir}")
        print("Procure pelos arquivos .safetensors nesta pasta.")
        print("=======================================================")

if __name__ == "__main__":
    main()

--- 0. LIMPEZA DE DISCO (DEEP CLEAN) ---
🗑️ Removendo lixo encontrado: /content/sample_data
⚙️ [INICIANDO]: Limpando Cache do PIP...
✅ [SUCESSO]: Limpando Cache do PIP

✅ Espaço em disco recuperado.
--- 1. DIAGNÓSTICO DE HARDWARE ---
--- 2. INSTALANDO BIBLIOTECAS (MODO COMPATIBILIDADE) ---
⚙️ [INICIANDO]: Clonando Diffusers...
✅ [SUCESSO]: Clonando Diffusers

⚙️ [INICIANDO]: Instalando Dependências Core...
✅ [SUCESSO]: Instalando Dependências Core

⚙️ [INICIANDO]: Instalando Diffusers (Source)...
✅ [SUCESSO]: Instalando Diffusers (Source)

--- 3. PREPARANDO IMAGENS ---
📂 Extraindo imagens.zip...
✅ 245 imagens prontas em /content/instance_data
--- 4. TREINANDO (Hugging Face Dreambooth LoRA) ---
🚀 Iniciando Neural Network...
⚙️ [INICIANDO]: Treinamento...
✅ [SUCESSO]: Treinamento


🎉 SUCESSO! Modelo salvo em: /content/output_lora
Procure pelos arquivos .safetensors nesta pasta.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
